# 🎥 LTX-2 Video Generation Server (L4 GPU)

Run **LTX-2.3** (Lightricks' DiT-based audio-video foundation model) on a **Google Colab L4 GPU** (22 GB VRAM)
and expose it as a **FastAPI server** via **ngrok**.

Uses the **DistilledPipeline** with **FP8 quantization** for fast inference that fits in 22 GB VRAM.

---

## ⚙️ Prerequisites — Colab Secrets

Before running, add these secrets in the Colab sidebar (🔑 icon → "Secrets"):

| Secret Name | Where to get it | What it does |
|---|---|---|
| `NGROK_TOKEN` | https://dashboard.ngrok.com/get-started/your-authtoken | Creates a public tunnel to your API |
| `HF_TOKEN` | https://huggingface.co/settings/tokens | Downloads model weights from HuggingFace |

> **Tip:** Toggle "Notebook access" ON for each secret after adding it.

---

## 📋 How to use

1. Select **Runtime → Change runtime type → L4 GPU**
2. Add your secrets (see above)
3. Run **Cell 1** — installs dependencies and downloads models (~50 GB, takes ~10-15 min)
4. Run **Cell 1b** — applies Gemma 3 compatibility patches (**required — fixes generation bug**)
5. Run **Cell 2** — loads model and starts the FastAPI + ngrok server
6. Copy the **ngrok URL** printed in the output
7. Send POST requests to `<ngrok_url>/generate` to generate videos

### Example request (Python)
```python
import requests

URL = "https://<your-ngrok-url>/generate"

resp = requests.post(URL, json={
    "prompt": "A golden retriever playing in ocean waves at sunset, slow motion, cinematic",
    "seed": 42,
    "height": 512,
    "width": 768,
    "num_frames": 97
}, timeout=600)

with open("output.mp4", "wb") as f:
    f.write(resp.content)
print("Video saved!")
```

### Example request (curl)
```bash
curl -X POST <ngrok_url>/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt": "A cat sitting on a windowsill watching rain", "seed": 42}' \
  --output output.mp4
```

---

## 📝 Notes
- **Model:** LTX-2.3 22B Distilled + Spatial Upscaler x2 + Gemma 3 12B (Q4)
- **Pipeline:** DistilledPipeline (8 steps stage 1, 4 steps stage 2 — fastest)
- **Quantization:** FP8 cast (fits in 22 GB VRAM)
- **Output:** MP4 video with synchronized audio
- **Default resolution:** 512×768 → upscaled to 1024×1536 (two-stage)
- **Default frames:** 97 (~4 seconds at 24 fps)

In [ ]:
# =============================================================================
# CELL 1 — Install Dependencies + Download Models
# =============================================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Install LTX-2 from source ---
print("🔧 Installing LTX-2...")
!git clone --depth 1 https://github.com/Lightricks/LTX-2.git /content/LTX-2 2>/dev/null || echo "Already cloned"
!pip install -q -e /content/LTX-2/packages/ltx-core -e /content/LTX-2/packages/ltx-pipelines 2>&1 | tail -5

# --- Install server dependencies ---
!pip install -q fastapi uvicorn pyngrok huggingface_hub hf_transfer 2>&1 | tail -3

# --- Download model weights ---
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from google.colab import userdata
from huggingface_hub import hf_hub_download, snapshot_download

HF_TOKEN = userdata.get("HF_TOKEN")
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("\n📦 Downloading LTX-2.3 distilled checkpoint...")
distilled_ckpt = hf_hub_download(
    repo_id="Lightricks/LTX-2.3",
    filename="ltx-2.3-22b-distilled.safetensors",
    local_dir=MODEL_DIR,
    token=HF_TOKEN,
)

print("📦 Downloading spatial upscaler (x2)...")
upsampler_path = hf_hub_download(
    repo_id="Lightricks/LTX-2.3",
    filename="ltx-2.3-spatial-upscaler-x2-1.0.safetensors",
    local_dir=MODEL_DIR,
    token=HF_TOKEN,
)

print("📦 Downloading Gemma 3 text encoder (Q4)...")
gemma_root = snapshot_download(
    repo_id="google/gemma-3-12b-it-qat-q4_0-unquantized",
    local_dir=os.path.join(MODEL_DIR, "gemma-3-12b-it-qat-q4_0-unquantized"),
    token=HF_TOKEN,
)

print(f"\n\u2705 All models downloaded!")
print(f"  Distilled checkpoint: {distilled_ckpt}")
print(f"  Spatial upsampler:    {upsampler_path}")
print(f"  Gemma root:           {gemma_root}")

In [ ]:
# =============================================================================
# CELL 1b — Patch LTX-2 for Gemma 3 Compatibility
# =============================================================================
# encoder_configurator.py has Gemma 3 incompatibilities:
#   1. config.rope_local_base_freq  → AttributeError (attr does not exist)
#   2. config.rope_scaling[...] / .get() → KeyError
#      The Gemma 3 config object's .get() raises KeyError for missing keys
#      instead of returning None (unlike a plain Python dict).
#
# This cell is idempotent — safe to re-run if Cell 2 still fails.

import os, re

enc_path = "/content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py"

with open(enc_path, "r") as f:
    src = f.read()

# ── Fix 1: rope_local_base_freq attribute ──────────────────────────────────
if "base = config.rope_local_base_freq" in src:
    src = src.replace(
        "base = config.rope_local_base_freq",
        'base = getattr(config, "rope_local_base_freq", 10000.0)',
    )
    print("  ↳ Fixed rope_local_base_freq")

# ── Fix 2: rope_type assignment → robust try/except ───────────────────────
# Replaces ANY "rope_type = ..." line with a safe block that handles:
#   • None  → fallback "linear"
#   • plain dict  → .get() is safe
#   • custom config object  → use getattr (avoids .get() KeyError)
# Guard: only apply if our sentinel '_rs = config.rope_scaling' is absent.
if "_rs = config.rope_scaling" not in src:
    _rope_re = re.compile(r'([ \t]+)rope_type\s*=\s*.+', re.MULTILINE)
    _rope_fix = (
        "\\1try:\n"
        "\\1    _rs = config.rope_scaling\n"
        "\\1    if _rs is None:\n"
        '\\1        rope_type = "linear"\n'
        "\\1    elif isinstance(_rs, dict):\n"
        '\\1        rope_type = _rs.get("rope_type") or _rs.get("type") or "linear"\n'
        "\\1    else:\n"
        '\\1        rope_type = (getattr(_rs, "rope_type", None) or getattr(_rs, "type", None) or "linear")\n'
        "\\1except Exception:\n"
        '\\1    rope_type = "linear"'
    )
    src = _rope_re.sub(_rope_fix, src, count=1)
    print("  ↳ Fixed rope_type lookup (try/except; handles dict & config objects)")

# ── Fix 3: ROPE_INIT_FUNCTIONS call → use rope_type variable ─────────────
# The original (and previously-patched) code inlines the rope_scaling lookup
# directly inside ROPE_INIT_FUNCTIONS[...], which triggers the same KeyError.
# Replace it with the rope_type variable computed by Fix 2.
if "ROPE_INIT_FUNCTIONS[rope_type]" not in src:
    src = re.sub(
        r'ROPE_INIT_FUNCTIONS\[.+?\]\(config\)',
        'ROPE_INIT_FUNCTIONS[rope_type](config)',
        src,
    )
    print("  ↳ Fixed ROPE_INIT_FUNCTIONS call to use rope_type variable")

# ── Fix 4: circular import guard (protects against runtime restart) ────────
loader_init = "/content/LTX-2/packages/ltx-core/src/ltx_core/loader/__init__.py"
if os.path.exists(loader_init):
    with open(loader_init, "r") as f:
        loader_src = f.read()
    patched_loader = loader_src.replace(
        "from ltx_core.loader.fuse_loras import apply_loras\n", ""
    )
    if patched_loader != loader_src:
        with open(loader_init, "w") as f:
            f.write(patched_loader)
        print("  ↳ Removed circular import from loader/__init__.py")

with open(enc_path, "w") as f:
    f.write(src)

print("✅ encoder_configurator.py patched successfully!")
print("\nNow run Cell 2 to load the model and start the server.")

In [ ]:
# =============================================================================
# CELL 2 — Load Model + Start FastAPI Server + ngrok Tunnel
# =============================================================================

import os, time, threading, tempfile, uuid, logging
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel, Field
from pyngrok import ngrok
from google.colab import userdata

from ltx_core.quantization import QuantizationPolicy
from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
from ltx_pipelines.distilled import DistilledPipeline
from ltx_pipelines.utils.media_io import encode_video

logging.getLogger().setLevel(logging.INFO)

# ── Config ──
MODEL_DIR = "/content/models"
DISTILLED_CKPT = os.path.join(MODEL_DIR, "ltx-2.3-22b-distilled.safetensors")
UPSAMPLER_PATH = os.path.join(MODEL_DIR, "ltx-2.3-spatial-upscaler-x2-1.0.safetensors")
GEMMA_ROOT = os.path.join(MODEL_DIR, "gemma-3-12b-it-qat-q4_0-unquantized")
OUTPUT_DIR = "/content/outputs"
API_PORT = 8080

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Load the pipeline with FP8 quantization ──
print("🔄 Loading LTX-2.3 DistilledPipeline with FP8 quantization...")
print("   (This takes a few minutes on first load)")

pipeline = DistilledPipeline(
    distilled_checkpoint_path=DISTILLED_CKPT,
    spatial_upsampler_path=UPSAMPLER_PATH,
    gemma_root=GEMMA_ROOT,
    loras=[],
    quantization=QuantizationPolicy.fp8_cast(),
)
print("✅ Pipeline loaded!")

# ── 2. FastAPI app ──
app = FastAPI(title="LTX-2 Video Generation Server")

# Lock to prevent concurrent generation (single GPU)
generation_lock = threading.Lock()


class GenerateRequest(BaseModel):
    prompt: str = Field(..., description="Text prompt describing the video to generate")
    seed: int = Field(default=42, description="Random seed for reproducibility")
    height: int = Field(default=512, description="Video height (will be 2x upscaled). Must be divisible by 32")
    width: int = Field(default=768, description="Video width (will be 2x upscaled). Must be divisible by 32")
    num_frames: int = Field(default=97, description="Number of frames. Must be 8*k+1 (e.g. 25, 49, 97, 121)")
    frame_rate: float = Field(default=24.0, description="Frame rate (fps)")


@app.get("/health")
async def health():
    return {"status": "ok", "model": "LTX-2.3-22B-Distilled", "quantization": "fp8-cast"}


@app.post("/generate")
async def generate(req: GenerateRequest):
    if not generation_lock.acquire(blocking=False):
        return JSONResponse(
            status_code=503,
            content={"error": "A generation is already in progress. Please wait and retry."},
        )
    try:
        video_id = str(uuid.uuid4())[:8]
        output_path = os.path.join(OUTPUT_DIR, f"{video_id}.mp4")

        print(f"\n🎥 Generating video {video_id}...")
        print(f"   Prompt: {req.prompt[:80]}{'...' if len(req.prompt) > 80 else ''}")
        print(f"   Resolution: {req.width}x{req.height} -> {req.width*2}x{req.height*2} (upscaled)")
        print(f"   Frames: {req.num_frames} @ {req.frame_rate} fps")

        start_time = time.time()

        with torch.inference_mode():
            tiling_config = TilingConfig.default()
            video_chunks_number = get_video_chunks_number(req.num_frames, tiling_config)

            video, audio = pipeline(
                prompt=req.prompt,
                seed=req.seed,
                height=req.height * 2,
                width=req.width * 2,
                num_frames=req.num_frames,
                frame_rate=req.frame_rate,
                images=[],
                tiling_config=tiling_config,
            )

            encode_video(
                video=video,
                fps=req.frame_rate,
                audio=audio,
                output_path=output_path,
                video_chunks_number=video_chunks_number,
            )

        elapsed = time.time() - start_time
        print(f"✅ Video generated in {elapsed:.1f}s -> {output_path}")

        return FileResponse(
            output_path,
            media_type="video/mp4",
            filename=f"ltx2_{video_id}.mp4",
            headers={"X-Generation-Time": f"{elapsed:.1f}s"},
        )
    except Exception as e:
        logging.exception("Generation failed")
        return JSONResponse(status_code=500, content={"error": str(e)})
    finally:
        generation_lock.release()


# ── 3. Start ngrok tunnel ──
NGROK_AUTH_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(API_PORT)
public_url = tunnel.public_url

# ── 4. Run FastAPI in background thread ──
server_thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": API_PORT, "log_level": "warning"},
    daemon=True,
)
server_thread.start()
time.sleep(2)

print("\n" + "=" * 65)
print(f"🌐 PUBLIC API URL: {public_url}")
print(f"🏠 LOCAL API URL:  http://127.0.0.1:{API_PORT}")
print(f"=" * 65)
print(f"\n  GET  {public_url}/health")
print(f"  POST {public_url}/generate")
print(f"  GET  {public_url}/docs   (interactive Swagger UI)")
print(f"\n✅ Server is running! Send requests to generate videos.")
print(f"\nExample:")
print(f'  curl -X POST {public_url}/generate \\')
print(f'    -H "Content-Type: application/json" \\')
print(f'    -d \'{{"prompt": "A cat watching rain from a window", "seed": 42}}\' \\')
print(f'    --output video.mp4')

In [ ]:
# =============================================================================
# CELL 3 — Generate a Video (Quick Test)
# =============================================================================

import requests
from IPython.display import Video, display

API_URL = "http://127.0.0.1:8080/generate"

print("🎥 Generating test video...")
resp = requests.post(API_URL, json={
    "prompt": (
        "A golden retriever puppy runs along a sandy beach at sunset. "
        "Waves crash gently in the background, spraying mist into warm golden light. "
        "The puppy's fur glows in the low sun as it splashes through shallow water. "
        "Camera follows at a low angle, tracking the dog in slow motion. "
        "Cinematic, warm color grading, shallow depth of field."
    ),
    "seed": 42,
    "height": 512,
    "width": 768,
    "num_frames": 97,
    "frame_rate": 24.0,
}, timeout=600)

if resp.status_code == 200:
    output_file = "/content/outputs/test_video.mp4"
    with open(output_file, "wb") as f:
        f.write(resp.content)
    gen_time = resp.headers.get("X-Generation-Time", "unknown")
    print(f"✅ Video saved to {output_file} (generation time: {gen_time})")
    display(Video(output_file, embed=True, width=640))
else:
    print(f"❌ Error: {resp.status_code} - {resp.text}")

In [ ]:
# =============================================================================
# CELL 4 — Image-to-Video Generation (Optional)
# =============================================================================
# Upload an image to Colab, then use it as conditioning for video generation.
#
# To use: upload an image via the Colab file browser, then set IMAGE_PATH below.

import requests, base64, json
from IPython.display import Video, display
from google.colab import files

# Upload an image
print("📁 Upload an image to use as conditioning:")
uploaded = files.upload()

if uploaded:
    IMAGE_PATH = "/content/" + list(uploaded.keys())[0]
    print(f"\n🖼️ Using image: {IMAGE_PATH}")
    print("🎥 Generating image-conditioned video...")

    # For image-to-video, we need to use the CLI directly since the
    # DistilledPipeline accepts images as ImageConditioningInput tuples.
    # Using subprocess to call the pipeline with image conditioning:
    import subprocess
    result = subprocess.run([
        "python", "-m", "ltx_pipelines.distilled",
        "--distilled-checkpoint-path", "/content/models/ltx-2.3-22b-distilled.safetensors",
        "--spatial-upsampler-path", "/content/models/ltx-2.3-spatial-upscaler-x2-1.0.safetensors",
        "--gemma-root", "/content/models/gemma-3-12b-it-qat-q4_0-unquantized",
        "--quantization", "fp8-cast",
        "--prompt", "A cinematic video inspired by the input image, smooth camera motion, high quality",
        "--image", IMAGE_PATH, "0", "1.0",
        "--output-path", "/content/outputs/i2v_output.mp4",
        "--seed", "42",
        "--num-frames", "97",
    ], capture_output=True, text=True,
       env={**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})

    print(result.stdout[-500:] if result.stdout else "")
    if result.returncode == 0:
        print("✅ Image-to-video complete!")
        display(Video("/content/outputs/i2v_output.mp4", embed=True, width=640))
    else:
        print(f"❌ Error: {result.stderr[-500:]}")
else:
    print("No image uploaded. Skipping.")